In [5]:
import numpy as np
import pandas as pd


In [6]:
def gini(y):
    classes = np.unique(y)
    impurity =1
    for c in classes:
        p = np.sum(y==c)/len(y)
        impurity -= p**2
    return impurity  

def best_split(x,y):
    best_gini = 999
    best_threshold = None
    best_feat = None

    for feat in range(x.shape[1]):
        thresholds = np.unique(x[:,feat])
        for thresh in thresholds:
            left = y[x[:,feat]<= thresh]
            right = y[x[:,feat]> thresh]
            if len(left) ==0 or len(right)==0:
                continue

            g = (len(left)*gini(left) + len(right)*gini(right))/len(y)
            if g< best_gini:
                best_gini = g
                best_feat = feat
                best_threshold = thresh
                
    return best_feat,best_threshold


In [7]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)


In [19]:
X = df[['Pclass','Sex','Age']].copy()
Y = df['Survived']

X['Age'] = X['Age'].fillna(X['Age'].median())
Y = Y.fillna(Y.median())
X.loc[:,'Sex'] = X['Sex'].map({'male':1,'female ':0}).astype('str')
X.Sex = X.Sex.astype('float64')
X.loc[:, 'Sex'] = X['Sex'].map({'male': 1, 'female': 0}).astype('float64')
x = X.to_numpy(dtype='float64')
y = Y.to_numpy(dtype='float64')


In [ ]:
np.random.seed(42)

split_lenght = np.round(0.8 * len(x)).astype(int)

x_train = x[np.random.permutation(len(x))[:split_lenght]]    
y_train = y[np.random.permutation(len(y))[:split_lenght]]
x_test = x[np.random.permutation(len(x))[split_lenght:]]    
y_test = y[np.random.permutation(len(y))[split_lenght:]]


In [ ]:
feat, thresh = best_split(x_train,y_train)

In [ ]:
print(f"Best split feat : {feat}, threshold : {thresh}")
print(f"left gini : {gini(y_train[x_train[:,feat]<=thresh]):.3f}")
print(f"right gini : {gini(y_train[x_train[:,feat]>thresh]):.3f}")

Best split feat : 1, threshold : 0.0
left gini : 0.383
right gini : 0.307


In [ ]:
def build_tree(x,y,depth=0,max_depth = 3):
    if gini(y)==0:
        return {"predict": int(y[0])}
    if depth ==max_depth:
        return {"predict": int(np.round(np.mean(y)))}
    
    feat , thresh = best_split(x,y)

    left_mask = x[:, feat] <= thresh
    right_mask = x[:, feat] > thresh
    print(f"{'  '*depth}depth {depth}: feature : {feat}, threshold :  {thresh}")

    return {
        "feat":   feat,
        "thresh": thresh,
        "left":   build_tree(x[left_mask],  y[left_mask],  depth+1, max_depth),
        "right":  build_tree(x[right_mask], y[right_mask], depth+1, max_depth),
    }

def predict_one(node,x):
    if 'predict' in node:
        return node["predict"]
    if x[node["feat"]] <= node["thresh"]:
        return predict_one(node["left"],x)
    else:
        return predict_one(node["right"],x)



In [ ]:
tree = build_tree(x_train,y_train)

depth 0: feature : 1, threshold :  0.0
  depth 1: feature : 0, threshold :  2.0
    depth 2: feature : 2, threshold :  2.0
    depth 2: feature : 2, threshold :  36.0
  depth 1: feature : 2, threshold :  6.0
    depth 2: feature : 0, threshold :  2.0
    depth 2: feature : 0, threshold :  1.0


In [ ]:
print("\n--- Predictions ---")
 
correct_prediction =0
for i, row in enumerate(x_test):
    pred = predict_one(tree, row)
    actual = y_test[i]
    status = "correct" if pred == actual else "WRONG"
    correct_prediction += 1 if status == 'correct' else 0
    print(f"Pclass={row[0]}, Sex={row[1]}, Age={row[2]} → pred={pred}, actual={actual} ({status})")
accuracy = correct_prediction/len(y_test)
print(round(accuracy*100,2),"% accurate")



--- Predictions ---
Pclass=3.0, Sex=1.0, Age=28.0 → pred=0, actual=1.0 (WRONG)
Pclass=2.0, Sex=1.0, Age=31.0 → pred=0, actual=0.0 (correct)
Pclass=3.0, Sex=1.0, Age=20.0 → pred=0, actual=0.0 (correct)
Pclass=2.0, Sex=0.0, Age=6.0 → pred=1, actual=1.0 (correct)
Pclass=3.0, Sex=0.0, Age=14.0 → pred=1, actual=1.0 (correct)
Pclass=1.0, Sex=0.0, Age=26.0 → pred=1, actual=1.0 (correct)
Pclass=3.0, Sex=0.0, Age=28.0 → pred=1, actual=1.0 (correct)
Pclass=3.0, Sex=1.0, Age=16.0 → pred=0, actual=0.0 (correct)
Pclass=3.0, Sex=0.0, Age=16.0 → pred=1, actual=1.0 (correct)
Pclass=1.0, Sex=0.0, Age=19.0 → pred=1, actual=1.0 (correct)
Pclass=1.0, Sex=1.0, Age=37.0 → pred=0, actual=0.0 (correct)
Pclass=3.0, Sex=1.0, Age=44.0 → pred=0, actual=0.0 (correct)
Pclass=3.0, Sex=0.0, Age=28.0 → pred=1, actual=0.0 (WRONG)
Pclass=3.0, Sex=1.0, Age=30.0 → pred=0, actual=0.0 (correct)
Pclass=2.0, Sex=1.0, Age=36.0 → pred=0, actual=0.0 (correct)
Pclass=1.0, Sex=0.0, Age=16.0 → pred=1, actual=1.0 (correct)
Pclass=1